# Jittering with WTI

In [ ]:
import pandas as pd
import numpy as np
import itertools
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
import importlib
import gc
import tensorflow as tf
from tensorflow.keras import backend as K
import my_lstm
importlib.reload(my_lstm)

from my_lstm import build_lstm_model, create_sequences, expanding_window_lstm_forecast3

In [ ]:
df = pd.read_csv('../data/final_data.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.head()

In [ ]:
feature_cols = [
    "wti_ret"
]

target_col = "wti_ret"

In [ ]:
df = df.sort_values("Date").reset_index(drop=True)

# data split into 70/15/15
train_size = int(len(df) * 0.7)
val_size = int(len(df) * 0.15)
val_end = train_size + val_size


# split using iloc 
train_data = df.iloc[:train_size]
val_data = df.iloc[train_size:val_end]
test_data = df.iloc[val_end:] 

# print length of data in each set
print(f'Train data length: {len(train_data)}')
print(f'Validation data length: {len(val_data)}')
print(f'Test data length: {len(test_data)}')

In [ ]:
param_grid_tiny = {
    "lookback": [2, 10],
    "dropout": [0.001, 0.1],
    "units": [50, 170],
    "epochs": [50, 100]
}

param_combinations = list(itertools.product(
    param_grid_tiny["lookback"],
    param_grid_tiny["dropout"],
    param_grid_tiny["units"],
    param_grid_tiny["epochs"]
))

results_grid = []
failed_combos = []

max_retries = 2

for i, (lb, dr, units, ep) in enumerate(param_combinations, 1):
    print(f"\n[{i}/{len(param_combinations)}] Testing params:")
    print(f"lookback={lb}, dropout={dr}, units={units}, epochs={ep}")

    success = False

    for attempt in range(1, max_retries + 1):
        try:
            print(f"Attempt {attempt}/{max_retries}")

            # clear memory before each attempt
            K.clear_session()
            tf.keras.backend.clear_session()
            gc.collect()

            val_forecasts = expanding_window_lstm_forecast3(
                df=df,
                feature_cols=feature_cols,
                target_col=target_col,
                initial_train_size=train_size,
                end_idx=val_end,
                date_col="Date",
                lookback=lb,
                units=units,
                dropout=dr,
                epochs=ep,
                batch_size=32,
                verbose=0,
                scale=True,
                seed=42,
                nojitter=False,          # use jitter
                jitter_std=0.03,         
                jitter_cols=feature_cols,
                num_samples=1              # generate 1 sample
            )

            if len(val_forecasts) == 0:
                print("No forecasts generated.")
                raise ValueError("No forecasts generated")

            mse = mean_squared_error(
                val_forecasts["actual"],
                val_forecasts["predicted"]
            )

            print(f"Validation MSE: {mse:.6f}")

            results_grid.append({
                "lookback": lb,
                "dropout": dr,
                "units": units,
                "epochs": ep,
                "mse": mse,
                "attempt_used": attempt
            })

            success = True
            break

        except Exception as e:
            print(f"Error on attempt {attempt}: {e}")

            # clear memory after failure 
            K.clear_session()
            tf.keras.backend.clear_session()
            gc.collect()

    if not success:
        print("Failed after all retries. Skipping this combination.")
        failed_combos.append({
            "lookback": lb,
            "dropout": dr,
            "units": units,
            "epochs": ep
        })

# --------------------------------------------------
# results
# --------------------------------------------------

results_grid_df = pd.DataFrame(results_grid).sort_values("mse").reset_index(drop=True)
failed_combos_df = pd.DataFrame(failed_combos)

In [ ]:
print("\nTop results:")
print(results_grid_df.head())

print("\nFailed combinations:")
print(failed_combos_df)

In [ ]:
best_params = results_grid_df.iloc[0]
best_lb = int(best_params["lookback"])
best_dr = float(best_params["dropout"])
best_units = int(best_params["units"])
best_ep = int(best_params["epochs"])

best_params

In [ ]:
test_results = expanding_window_lstm_forecast3(
    df=df,
    feature_cols=feature_cols,
    target_col=target_col,
    initial_train_size=val_end,
    end_idx=len(df),
    date_col="Date",
    lookback=int(best_params["lookback"]),
    units=int(best_params["units"]),
    dropout=float(best_params["dropout"]),
    epochs=int(best_params["epochs"]),
    batch_size=32,
    verbose=0,
    scale=True,
    seed=42,
    nojitter=False,          
    jitter_std=0.03,         
    jitter_cols=feature_cols,
    num_samples=1            # generate 1 sample
)

test_mse = mean_squared_error(
    test_results["actual"],
    test_results["predicted"]
)

test_mape = mean_absolute_percentage_error(
    test_results["actual"],
    test_results["predicted"]
)

test_r2 = r2_score(
    test_results["actual"],
    test_results["predicted"]
)

print("Test MSE:", test_mse)
print("Test MAPE:", test_mape)
print("Test R²:", test_r2)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(test_results['Date'], test_results["actual"], label="Actual")
plt.plot(test_results['Date'], test_results["predicted"], label="Predicted")
plt.legend()
plt.title("Out-of-sample Forecast for LSTM-JITTERING (TB3M)")
plt.show()

In [ ]:
# save results to csv
test_results.to_csv('results/lstm_update_jitter_results.csv', index = False)